# Field Proposal Review Risk Model PoC

기존 합성 상담 시나리오에서 로컬로 생성한 150행 CSV를 검증하고 Dummy, Logistic Regression, Random Forest를 학습·평가합니다. 원천 JSONL과 실제 개인정보는 Colab에 업로드하지 않습니다. `needs_review`와 `confidence`는 제출용 PoC를 위해 파생·모사한 값이며 실서비스 성능을 의미하지 않습니다.

In [1]:
# @title 1. 실행 경로 및 라이브러리 확인
from pathlib import Path
import os
import importlib.util
import platform
import pandas as pd
import numpy as np
import sklearn
import matplotlib
import joblib

local_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
candidates = [Path(os.environ.get('F2_POC_ROOT', '')), Path('/content/f2_poc'), local_root, local_root / 'ml/field_proposal_reliability']
POC_ROOT = next((path.resolve() for path in candidates if path and (path / 'data/synthetic_field_proposals.csv').is_file() and (path / 'scripts/run_experiment.py').is_file()), None)
if POC_ROOT is None:
    raise FileNotFoundError('PoC root를 찾지 못했습니다. /content/f2_poc에 CSV와 run_experiment.py를 업로드하거나 F2_POC_ROOT를 설정하세요.')
print('PoC root:', POC_ROOT)
try:
    ON_GOOGLE_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    ON_GOOGLE_COLAB = False
RUNTIME_LABEL = 'google_colab_cpu' if ON_GOOGLE_COLAB else 'local_notebook_verification'
print('Runtime label:', RUNTIME_LABEL)
print('Python:', platform.python_version())
print('pandas:', pd.__version__)
print('numpy:', np.__version__)
print('scikit-learn:', sklearn.__version__)
print('matplotlib:', matplotlib.__version__)
print('joblib:', joblib.__version__)

PoC root: /home/hong/project/ai-camp-project-note/projects/SKN30-FINAL-3Team/ml/field_proposal_reliability
Runtime label: local_notebook_verification
Python: 3.14.3
pandas: 3.0.5
numpy: 2.5.2
scikit-learn: 1.9.0
matplotlib: 3.11.1
joblib: 1.5.3


In [2]:
# @title 2. 데이터 기본 통계 확인
dataset_path = POC_ROOT / 'data/synthetic_field_proposals.csv'
df = pd.read_csv(dataset_path)
print('전체 행 수:', len(df))
print('컬럼 목록:', df.columns.tolist())
print('Target 분포:')
print(df['needs_review'].value_counts().sort_index())
print('field_type 분포:')
print(df['field_type'].value_counts().sort_index())
print('결측치 개수:')
print(df.isna().sum())
print('첫 10행:')
display(df.head(10))

전체 행 수: 150
컬럼 목록: ['proposal_id', 'source_scenario_id', 'source_group_id', 'source_transcript_hash', 'source_dataset', 'source_label', 'proxy_risk_score', 'field_type', 'confidence', 'evidence_length', 'mention_count', 'has_conflict', 'has_negation', 'parse_success', 'needs_review']
Target 분포:
needs_review
0    80
1    70
Name: count, dtype: int64
field_type 분포:
field_type
building_number       15
deal_type             15
expiry_date           15
handover_condition    15
jeonse_deposit        15
monthly_deposit       15
monthly_rent          15
pyeong                15
sale_price            15
unit_number           15
Name: count, dtype: int64
결측치 개수:
proposal_id               0
source_scenario_id        0
source_group_id           0
source_transcript_hash    0
source_dataset            0
source_label              0
proxy_risk_score          0
field_type                0
confidence                0
evidence_length           0
mention_count             0
has_conflict              0
has

,proposal_id,source_scenario_id,source_group_id,source_transcript_hash,source_dataset,source_label,proxy_risk_score,field_type,confidence,evidence_length,mention_count,has_conflict,has_negation,parse_success,needs_review
0,f2-review-sale_price-01,f2-sell-request-safe-0037,sell-safe-blueprint-0037,583a22e08e6f47485c7e1d4f389c0288f7a8ad4a04074d...,f2_sllm_blueprint,매도의뢰,2.003345,sale_price,0.741330,32,2,1,0,1,0
1,f2-review-sale_price-02,f2-sell-request-safe-0018,sell-safe-blueprint-0018,5f3004978cf03a6964d070055d439d26d40120de744fc6...,f2_sllm_blueprint,매도의뢰,3.494134,sale_price,0.692532,45,4,1,0,1,1
2,f2-review-sale_price-03,f2-buy-request-safe-0049,buy-safe-blueprint-0049,f29a7303e0097333a3491e103afeef22ec90218d97d1c7...,f2_sllm_blueprint,매수문의,3.787382,sale_price,0.503428,32,2,1,0,0,1
3,f2-review-sale_price-04,f2-sell-request-safe-0046,sell-safe-blueprint-0046,d6225f45b102cec427bd0c970f03da255abb792c8ff634...,f2_sllm_blueprint,매도의뢰,2.281417,sale_price,0.728949,20,2,1,0,1,0
4,f2-review-sale_price-05,f2-sell-request-safe-0010,sell-safe-blueprint-0010,1fac99cd50c9592a6022c463b77116f18b8c5393b4b172...,f2_sllm_blueprint,매도의뢰,2.543787,sale_price,0.678824,36,3,1,0,1,0
5,f2-review-sale_price-06,f2-sell-request-safe-0021,sell-safe-blueprint-0021,470065d3ad1a5476b23210615a0263bddb03ccd10984ae...,f2_sllm_blueprint,매도의뢰,3.419111,sale_price,0.701558,26,4,1,0,1,1
6,f2-review-sale_price-07,f2-sell-request-safe-0041,sell-safe-blueprint-0041,78fb4a69328bf2b7fa6418d4baf8ec7101e568baeae15f...,f2_sllm_blueprint,매도의뢰,3.225290,sale_price,0.684622,34,3,1,0,1,0
7,f2-review-sale_price-08,f2-sell-request-safe-0013,sell-safe-blueprint-0013,fde9d9ce50e65da956cc9014d52884cb0de25964df073a...,f2_sllm_blueprint,매도의뢰,2.473796,sale_price,0.712726,19,3,1,0,1,0
8,f2-review-sale_price-09,f2-buy-request-safe-0012,buy-safe-blueprint-0012,cc3c21f7ee0b78ef891c02b3131cfa51758a291a65b973...,f2_sllm_blueprint,매수문의,4.982766,sale_price,0.565813,38,2,1,1,0,1
9,f2-review-sale_price-10,f2-co-brokerage-safe-0001,co-brokerage-safe-blueprint-0001,8cec54355e888a916847d4504d4036bdbdce4f273bd436...,f2_sllm_blueprint,공동중개,4.579543,sale_price,0.701492,47,2,1,0,0,1


In [3]:
# @title 3. 모델 학습, 평가, 저장
import subprocess
import sys
command = [sys.executable, str(POC_ROOT / 'scripts/run_experiment.py'), '--poc-root', str(POC_ROOT), '--dataset', str(dataset_path), '--execution-label', RUNTIME_LABEL]
completed = subprocess.run(command, text=True, capture_output=True)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr, file=sys.stderr)
completed.check_returncode()

작업 완료

Dataset
- 행 수: 150
- Train: 120
- Test: 30
- Positive class 비율: 0.4667

Logistic Regression
- Accuracy: 0.8000
- Precision: 0.9000
- Recall: 0.6429
- F1: 0.7500

Random Forest
- Accuracy: 0.8000
- Precision: 0.9000
- Recall: 0.6429
- F1: 0.7500

Final Model
- 선정 모델: Logistic Regression
- 선정 이유: Recall과 F1이 같아 Train/Test 성능 차이가 더 작은 Logistic Regression을 선택하였다.

Generated Files
- /home/hong/project/ai-camp-project-note/projects/SKN30-FINAL-3Team/ml/field_proposal_reliability/reports/eda_summary.json
- /home/hong/project/ai-camp-project-note/projects/SKN30-FINAL-3Team/ml/field_proposal_reliability/reports/model_comparison.csv
- /home/hong/project/ai-camp-project-note/projects/SKN30-FINAL-3Team/ml/field_proposal_reliability/reports/metrics.json
- /home/hong/project/ai-camp-project-note/projects/SKN30-FINAL-3Team/ml/field_proposal_reliability/reports/confusion_matrix_logistic_regression.png
- /home/hong/project/ai-camp-project-note/projects/SKN30-FINAL-3Team/ml/field_proposal_reliabi

In [4]:
# @title 4. 실제 결과 확인 및 다운로드 묶음 생성
import json
import shutil
metrics = json.loads((POC_ROOT / 'reports/metrics.json').read_text(encoding='utf-8'))
comparison = pd.read_csv(POC_ROOT / 'reports/model_comparison.csv')
display(comparison)
print('Final Model:', metrics['final_model']['name'])
print('선정 이유:', metrics['final_model']['selection_reason'])
archive_base = Path('/content/f2_poc_outputs') if ON_GOOGLE_COLAB else Path('/tmp/f2_poc_outputs')
archive = shutil.make_archive(str(archive_base), 'zip', root_dir=POC_ROOT)
print('다운로드 파일:', archive)

,Model,Train Accuracy,Test Accuracy,Precision,Recall,F1
0,Dummy,0.533333,0.533333,0.0,0.000000,0.00
1,Logistic Regression,0.791667,0.800000,0.9,0.642857,0.75
2,Random Forest,0.975000,0.800000,0.9,0.642857,0.75


Final Model: Logistic Regression
선정 이유: Recall과 F1이 같아 Train/Test 성능 차이가 더 작은 Logistic Regression을 선택하였다.
다운로드 파일: /tmp/f2_poc_outputs.zip
